In [ ]:
import cv2
import numpy as np

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
INTERP_T = 0.5
FLOW_SCALE = 1.0     # change to 1/32768 if PL outputs Q1.15 flow

# ------------------------------------------------------------
# WARP FUNCTION
# ------------------------------------------------------------
def warp_frame(frame, flow, direction):
    h, w = frame.shape[:2]

    dx = flow[0].astype(np.float32) * FLOW_SCALE * direction
    dy = flow[1].astype(np.float32) * FLOW_SCALE * direction

    x, y = np.meshgrid(
        np.arange(w, dtype=np.float32),
        np.arange(h, dtype=np.float32)
    )

    map_x = x + dx
    map_y = y + dy

    warped = cv2.remap(
        frame,
        map_x,
        map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )

    return warped

# ------------------------------------------------------------
# POST-PROCESS + VIDEO WRITE
# ------------------------------------------------------------
def postprocess_and_write(
    frame0_full,
    frame1_full,
    flow,
    video_writer,
    resize_h=224,
    resize_w=224
):
    """
    frame0_full, frame1_full : uint8 (H0,W0,3)
    flow : (2,224,224) int16
    video_writer : cv2.VideoWriter
    """

    # 1️⃣ Write original frame
    video_writer.write(frame0_full)

    # 2️⃣ Resize frames to flow resolution
    f0 = cv2.resize(frame0_full, (resize_w, resize_h))
    f1 = cv2.resize(frame1_full, (resize_w, resize_h))

    # 3️⃣ Warp
    w0 = warp_frame(f0, flow, +INTERP_T)
    w1 = warp_frame(f1, flow, -(1.0 - INTERP_T))

    # 4️⃣ Blend
    interp_small = (
        (1.0 - INTERP_T) * w0.astype(np.float32) +
        INTERP_T * w1.astype(np.float32)
    )

    interp_small = np.clip(interp_small, 0, 255).astype(np.uint8)

    # 5️⃣ Upscale to original resolution
    h0, w0 = frame0_full.shape[:2]
    interp_full = cv2.resize(
        interp_small,
        (w0, h0),
        interpolation=cv2.INTER_LINEAR
    )

    # 6️⃣ Write interpolated frame
    video_writer.write(interp_full)